In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import re

catalog_name = "workspace"
schema_name = "default"

bronze_schema = f"{catalog_name}.{schema_name}"
silver_schema = f"{catalog_name}.{schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

def normalize_text(col_name):
    return F.trim(F.regexp_replace(F.col(col_name), r"\s+", " "))

def normalize_key(col_name):
    """
    Lowercase matching key for joins and dedupe.
    """
    return F.trim(
        F.regexp_replace(
            F.lower(F.col(col_name)),
            r"[^a-z0-9]+",
            " "
        )
    )

def parse_bool(col_name):
    col_str = F.lower(F.trim(F.col(col_name).cast("string")))
    
    return (
        F.when(col_str.isin("1", "true", "yes", "y"), F.lit(True))
         .when(col_str.isin("0", "false", "no", "n"), F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )

def safe_int(col_name):
    return F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.col(col_name).cast("int"))

def safe_double(col_name):
    return F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.col(col_name).cast("double"))

def clean_string_list(col_name):
    """
    Turns stringified list-like text into an array.
    Example:
    "['A', 'B']" -> ['A','B']
    """
    return F.when(
        F.col(col_name).isNull(), F.array().cast("array<string>")
    ).otherwise(
        F.split(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(F.col(col_name), r"^\[|\]$", ""),
                    r"'", ""
                ),
                r'"', ""
            ),
            r"\s*,\s*"
        )
    )

def add_silver_metadata(df, source_table_name):
    return (
        df.withColumn("silver_processed_ts", F.current_timestamp())
          .withColumn("silver_source_table", F.lit(source_table_name))
    )

def safe_long(col_name):
    col_str = F.trim(F.col(col_name).cast("string"))
    return F.when(
        (col_str == "") | col_str.isNull(),
        None
    ).otherwise(col_str.cast("double").cast("bigint"))

In [0]:
# -----------------------------------------
# READ BRONZE sephora_product_info
# -----------------------------------------
df = spark.table(f"{bronze_schema}.bronze_sephora_product_info")

# -----------------------------------------
# BASIC CLEANING
# -----------------------------------------
df_clean = (
    df
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", normalize_text("product_name"))
    .withColumn("brand_name", normalize_text("brand_name"))
    .withColumn("brand_id", safe_int("brand_id"))
    .withColumn("loves_count", safe_int("loves_count"))
    .withColumn("rating", safe_double("rating"))
    .withColumn("reviews", safe_int("reviews"))
    .withColumn("size", normalize_text("size"))
    .withColumn("variation_type", normalize_text("variation_type"))
    .withColumn("variation_value", normalize_text("variation_value"))
    .withColumn("variation_desc", normalize_text("variation_desc"))
    .withColumn("ingredients", normalize_text("ingredients"))
    .withColumn("price_usd", safe_double("price_usd"))
    .withColumn("value_price_usd", safe_double("value_price_usd"))
    .withColumn("sale_price_usd", safe_double("sale_price_usd"))
    .withColumn("limited_edition", parse_bool("limited_edition"))
    .withColumn("new", parse_bool("new"))
    .withColumn("online_only", parse_bool("online_only"))
    .withColumn("out_of_stock", parse_bool("out_of_stock"))
    .withColumn("sephora_exclusive", parse_bool("sephora_exclusive"))
    .withColumn("highlights", normalize_text("highlights"))
    .withColumn("primary_category", normalize_text("primary_category"))
    .withColumn("secondary_category", normalize_text("secondary_category"))
    .withColumn("tertiary_category", normalize_text("tertiary_category"))
    .withColumn("child_count", safe_int("child_count"))
    .withColumn("child_max_price", safe_double("child_max_price"))
    .withColumn("child_min_price", safe_double("child_min_price"))
)

# -----------------------------------------
# RENAME FOR CONSISTENCY
# -----------------------------------------
df_clean = (
    df_clean
    .withColumnRenamed("reviews", "review_count")
    .withColumnRenamed("new", "is_new")
    .withColumnRenamed("online_only", "is_online_only")
)

# -----------------------------------------
# DERIVED COLUMNS
# -----------------------------------------
df_clean = (
    df_clean
    .withColumn("current_price_usd", F.coalesce(F.col("sale_price_usd"), F.col("price_usd")))
    .withColumn("brand_name_std", normalize_key("brand_name"))
    .withColumn("product_name_std", normalize_key("product_name"))
    .withColumn(
        "brand_product_key",
        F.concat_ws("||", F.col("brand_name_std"), F.col("product_name_std"))
    )
    .withColumn(
        "category_path",
        F.concat_ws(" > ", "primary_category", "secondary_category", "tertiary_category")
    )
    .withColumn("ingredients_array", clean_string_list("ingredients"))
    .withColumn("highlights_array", clean_string_list("highlights"))
    .withColumn("ingredient_count", F.size("ingredients_array"))
    .withColumn("highlight_count", F.size("highlights_array"))
)

# -----------------------------------------
# OPTIONAL: SKINCARE FLAG
# -----------------------------------------
df_clean = df_clean.withColumn(
    "is_skincare_product",
    F.when(
        (F.lower(F.col("primary_category")).like("%skincare%")) |
        (F.lower(F.col("secondary_category")).like("%skincare%")) |
        (F.lower(F.col("tertiary_category")).like("%skincare%")),
        F.lit(True)
    ).otherwise(F.lit(False))
)

# -----------------------------------------
# DATA QUALITY FILTERS / FLAGS
# -----------------------------------------
df_clean = (
    df_clean
    .withColumn("dq_missing_product_id", F.col("product_id").isNull())
    .withColumn("dq_missing_product_name", F.col("product_name").isNull() | (F.col("product_name") == ""))
    .withColumn("dq_invalid_rating", (F.col("rating") < 0) | (F.col("rating") > 5))
    .withColumn("dq_invalid_price", F.col("current_price_usd") < 0)
)

# -----------------------------------------
# DEDUPLICATION
# Keep most recent Bronze ingestion if duplicates by product_id
# -----------------------------------------
window_spec = Window.partitionBy("product_id").orderBy(F.col("bronze_ingestion_ts").desc())

df_dedup = (
    df_clean
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# -----------------------------------------
# ADD SILVER METADATA
# -----------------------------------------
df_final = add_silver_metadata(df_dedup, "bronze_sephora_product_info")

# -----------------------------------------
# WRITE SILVER
# -----------------------------------------
df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_sephora_product_info")

display(df_final.limit(10))

In [0]:
# -----------------------------------------
# READ BRONZE skincare_cosmetic_p
# -----------------------------------------
df = spark.table(f"{bronze_schema}.bronze_skincare_cosmetic_p")

# -----------------------------------------
# CLEAN / RENAME
# -----------------------------------------
df_clean = (
    df
    .withColumnRenamed("label", "product_type_label")
    .withColumnRenamed("brand", "brand_name")
    .withColumnRenamed("name", "product_name")
    .withColumnRenamed("price", "price_usd")
    .withColumnRenamed("rank", "rating")
    .withColumnRenamed("combination", "is_combination_skin")
    .withColumnRenamed("dry", "is_dry_skin")
    .withColumnRenamed("normal", "is_normal_skin")
    .withColumnRenamed("oily", "is_oily_skin")
    .withColumnRenamed("sensitive", "is_sensitive_skin")
)

df_clean = (
    df_clean
    .withColumn("product_type_label", normalize_text("product_type_label"))
    .withColumn("brand_name", normalize_text("brand_name"))
    .withColumn("product_name", normalize_text("product_name"))
    .withColumn("price_usd", safe_double("price_usd"))
    .withColumn("rating", safe_double("rating"))
    .withColumn("ingredients", normalize_text("ingredients"))
    .withColumn("is_combination_skin", parse_bool("is_combination_skin"))
    .withColumn("is_dry_skin", parse_bool("is_dry_skin"))
    .withColumn("is_normal_skin", parse_bool("is_normal_skin"))
    .withColumn("is_oily_skin", parse_bool("is_oily_skin"))
    .withColumn("is_sensitive_skin", parse_bool("is_sensitive_skin"))
)

# -----------------------------------------
# DERIVED COLUMNS
# -----------------------------------------
df_clean = (
    df_clean
    .withColumn("brand_name_std", normalize_key("brand_name"))
    .withColumn("product_name_std", normalize_key("product_name"))
    .withColumn(
        "brand_product_key",
        F.concat_ws("||", F.col("brand_name_std"), F.col("product_name_std"))
    )
    .withColumn("ingredients_array", F.split(F.col("ingredients"), r"\s*,\s*"))
    .withColumn("ingredient_count", F.size("ingredients_array"))
    .withColumn(
        "supported_skin_types",
        F.array_remove(
            F.array(
                F.when(F.col("is_combination_skin") == True, F.lit("combination")),
                F.when(F.col("is_dry_skin") == True, F.lit("dry")),
                F.when(F.col("is_normal_skin") == True, F.lit("normal")),
                F.when(F.col("is_oily_skin") == True, F.lit("oily")),
                F.when(F.col("is_sensitive_skin") == True, F.lit("sensitive"))
            ),
            None
        )
    )
)

# -----------------------------------------
# DATA QUALITY FLAGS
# -----------------------------------------
df_clean = (
    df_clean
    .withColumn("dq_missing_brand_name", F.col("brand_name").isNull() | (F.col("brand_name") == ""))
    .withColumn("dq_missing_product_name", F.col("product_name").isNull() | (F.col("product_name") == ""))
    .withColumn("dq_invalid_rating", (F.col("rating") < 0) | (F.col("rating") > 5))
    .withColumn("dq_invalid_price", F.col("price_usd") < 0)
)

# -----------------------------------------
# DEDUPLICATION
# Keep best row by completeness within same brand/product
# -----------------------------------------
df_clean = df_clean.withColumn(
    "completeness_score",
    F.expr("""
        int(product_type_label is not null) +
        int(price_usd is not null) +
        int(rating is not null) +
        int(ingredients is not null) +
        int(size(ingredients_array) > 0)
    """)
)

window_spec = Window.partitionBy("brand_product_key").orderBy(
    F.col("completeness_score").desc(),
    F.col("bronze_ingestion_ts").desc()
)

df_dedup = (
    df_clean
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "completeness_score")
)

# -----------------------------------------
# ADD SILVER METADATA
# -----------------------------------------
df_final = add_silver_metadata(df_dedup, "bronze_skincare_cosmetic_p")

# -----------------------------------------
# WRITE SILVER
# -----------------------------------------
df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_skincare_cosmetic_p")

display(df_final.limit(10))

In [0]:
df = spark.table(f"{bronze_schema}.bronze_sephora_reviews")
print(df.columns)
display(df.limit(5))

In [0]:
# -----------------------------------------
# READ BRONZE sephora_reviews
# -----------------------------------------

df = spark.table(f"{bronze_schema}.bronze_sephora_reviews")

# -----------------------------------------
# BASIC NORMALIZATION
# Adjust these names if your Bronze review columns differ slightly
# -----------------------------------------
df_clean = df

# Common expected columns in the Sephora review data
possible_text_cols = [
    "author_id", "rating", "is_recommended", "helpfulness",
    "total_feedback_count", "total_neg_feedback_count",
    "total_pos_feedback_count", "submission_time", "review_text",
    "review_title", "skin_tone", "eye_color", "skin_type", "hair_color",
    "product_id", "product_name", "brand_name", "price_usd"
]

for c in possible_text_cols:
    if c in df_clean.columns:
        df_clean = df_clean.withColumn(c, normalize_text(c))

# -----------------------------------------
# CAST TYPES WHERE AVAILABLE
# -----------------------------------------
if "author_id" in df_clean.columns:
    df_clean = df_clean.withColumn("author_id", F.trim(F.col("author_id").cast("string")))

if "product_id" in df_clean.columns:
    df_clean = df_clean.withColumn("product_id", F.trim(F.col("product_id").cast("string")))

if "rating" in df_clean.columns:
    df_clean = df_clean.withColumn("rating", safe_double("rating"))

if "is_recommended" in df_clean.columns:
    df_clean = df_clean.withColumn("is_recommended", parse_bool("is_recommended"))

for col_name in ["helpfulness", "total_feedback_count", "total_neg_feedback_count", "total_pos_feedback_count"]:
    if col_name in df_clean.columns:
        df_clean = df_clean.withColumn(col_name, safe_long(col_name))

if "price_usd" in df_clean.columns:
    df_clean = df_clean.withColumn("price_usd", safe_double("price_usd"))

# -----------------------------------------
# DATE PARSING
# -----------------------------------------
if "submission_time" in df_clean.columns:
    df_clean = (
        df_clean
        .withColumn("review_ts", F.to_timestamp("submission_time"))
        .withColumn("review_date", F.to_date("submission_time"))
        .withColumn("review_year", F.year("review_date"))
        .withColumn("review_month", F.month("review_date"))
    )

# -----------------------------------------
# STANDARDIZED TEXT KEYS
# -----------------------------------------
if "brand_name" in df_clean.columns:
    df_clean = df_clean.withColumn("brand_name_std", normalize_key("brand_name"))

if "product_name" in df_clean.columns:
    df_clean = df_clean.withColumn("product_name_std", normalize_key("product_name"))

if "review_text" in df_clean.columns:
    df_clean = (
        df_clean
        .withColumn("review_text", normalize_text("review_text"))
        .withColumn("review_text_length", F.length("review_text"))
        .withColumn("review_text_hash", F.sha2(F.coalesce(F.col("review_text"), F.lit("")), 256))
    )

if "review_title" in df_clean.columns:
    df_clean = df_clean.withColumn("review_title", normalize_text("review_title"))

# -----------------------------------------
# DATA QUALITY FLAGS
# -----------------------------------------
if "rating" in df_clean.columns:
    df_clean = df_clean.withColumn("dq_invalid_rating", (F.col("rating") < 0) | (F.col("rating") > 5))

if "product_id" in df_clean.columns:
    df_clean = df_clean.withColumn("dq_missing_product_id", F.col("product_id").isNull())

if "review_text" in df_clean.columns:
    df_clean = df_clean.withColumn(
        "dq_missing_review_text",
        F.col("review_text").isNull() | (F.col("review_text") == "")
    )

# -----------------------------------------
# DEDUPLICATION
# Use strongest available composite key
# -----------------------------------------
dedupe_cols = [c for c in ["product_id", "author_id", "review_text_hash", "review_date"] if c in df_clean.columns]

if len(dedupe_cols) > 0:
    window_spec = Window.partitionBy(*dedupe_cols).orderBy(F.col("bronze_ingestion_ts").desc())
    df_clean = (
        df_clean
        .withColumn("rn", F.row_number().over(window_spec))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

# -----------------------------------------
# ADD SILVER METADATA
# -----------------------------------------
df_final = add_silver_metadata(df_clean, "bronze_sephora_reviews")

# -----------------------------------------
# WRITE SILVER
# -----------------------------------------
df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_sephora_reviews")

display(df_final.limit(10))

In [0]:
# -----------------------------------------
# silver_ingredients_exploded
# -----------------------------------------
# Explode ingredients so each ingredient becomes its own row, making filtering, counting, joining, and product similarity analysis easier.
df_sephora = spark.table(f"{silver_schema}.silver_sephora_product_info") \
    .select(
        F.lit("sephora").alias("source_system"),
        F.col("product_id").cast("string").alias("source_product_id"),
        "brand_name",
        "product_name",
        "brand_product_key",
        "ingredients_array"
    )

df_cosmetic = spark.table(f"{silver_schema}.silver_skincare_cosmetic_p") \
    .select(
        F.lit("cosmetic_p").alias("source_system"),
        F.col("brand_product_key").alias("source_product_id"),
        "brand_name",
        "product_name",
        "brand_product_key",
        "ingredients_array"
    )

df_all = df_sephora.unionByName(df_cosmetic)

df_exploded = (
    df_all
    .withColumn("ingredient_name_raw", F.explode_outer("ingredients_array"))
    .withColumn("ingredient_name_raw", F.trim(F.col("ingredient_name_raw")))
    .withColumn(
        "ingredient_name_std",
        F.trim(F.regexp_replace(F.lower(F.col("ingredient_name_raw")), r"[^a-z0-9]+", " "))
    )
    .filter(F.col("ingredient_name_raw").isNotNull() & (F.col("ingredient_name_raw") != ""))
    .withColumn("silver_processed_ts", F.current_timestamp())
)

df_exploded.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.silver_ingredients_exploded")

display(df_exploded.limit(20))

In [0]:
# -----------------------------------------
# CREATE MF-READY SILVER INTERACTIONS
# Grain = 1 row per user-product pair
# Used later by Gold + Matrix Factorization
# -----------------------------------------

df_reviews = spark.table(f"{silver_schema}.silver_sephora_reviews")

silver_mf_interactions = (
    df_reviews
    .filter(F.col("author_id").isNotNull())
    .filter(F.col("product_id").isNotNull())
    .filter(F.col("rating").isNotNull())
    .withColumn("rating", F.col("rating").cast("double"))
    .filter((F.col("rating") >= 1.0) & (F.col("rating") <= 5.0))
    .groupBy(
        F.col("author_id").cast("string").alias("user_id"),
        F.col("product_id").cast("string").alias("item_id")
    )
    .agg(
        F.avg("rating").alias("rating"),
        F.max("submission_time").alias("latest_submission_time")
    )
)

silver_mf_interactions.write.mode("overwrite").format("delta").saveAsTable(
    f"{silver_schema}.silver_mf_interactions"
)

display(silver_mf_interactions.limit(10))
print("Silver MF interactions:", silver_mf_interactions.count())

In [0]:
%sql
SHOW TABLES IN workspace.default

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.silver_sephora_product_info

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.silver_skincare_cosmetic_p

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.silver_sephora_reviews

In [0]:
%sql
SELECT is_skincare_product, COUNT(*)
FROM workspace.default.silver_sephora_product_info
GROUP BY is_skincare_product

In [0]:
%sql
SELECT brand_name, product_name, ingredient_count
FROM workspace.default.silver_skincare_cosmetic_p
LIMIT 20

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.silver_mf_interactions;